# Importing and exporting data

## Learning objectives

By the end of this notebook you will be able to:

- read CSV, TSV and Excel files, including sheets
- control column types, missing value codes and dates when reading
- read only the columns or rows you need
- export results to CSV and Excel in a processed-data folder

## Introduction

Most analyses start by reading a file. `read_csv` has many options, and using the right ones while
reading saves a lot of cleaning afterwards. The rule: **never edit raw data by hand**. Read the raw file,
fix it in code, and write the result to a separate processed file.

In [1]:
import pandas as pd
import numpy as np
RAW = "../datasets/raw/"
PROCESSED = "../datasets/processed/"

## CSV files

In [2]:
patients = pd.read_csv(RAW + "patients.csv")
patients.head()

,patient_id,site,sex,age,weight_kg,height_cm,smoker,group,enrolled
0,P0001,Site A,M,41,66.4,156.3,No,Treatment,2025-03-31
1,P0002,Site B,M,83,69.5,168.0,No,Control,2025-08-19
2,P0003,Site B,F,36,71.4,172.2,No,Control,2025-08-11
3,P0004,Site A,F,24,66.0,158.9,Yes,Control,2025-06-24
4,P0005,Site A,M,77,80.5,163.7,No,Control,2025-08-05


## Useful read_csv options

| option | does |
|---|---|
| `sep` | separator: `","`, `"\t"`, `";"` |
| `usecols` | read only some columns |
| `nrows` | read only the first n rows |
| `dtype` | force column types, e.g. `{"patient_id": str}` |
| `na_values` | extra codes that mean missing |
| `parse_dates` | convert columns to dates |
| `index_col` | use a column as the index |
| `skiprows`, `header` | deal with notes above the table |
| `encoding` | for files with special characters (`"utf-8"`, `"latin-1"`) |

In [3]:
small = pd.read_csv(RAW + "patients.csv", usecols=["patient_id", "site", "age"], nrows=5)
small

,patient_id,site,age
0,P0001,Site A,41
1,P0002,Site B,83
2,P0003,Site B,36
3,P0004,Site A,24
4,P0005,Site A,77


### Dates

Without `parse_dates`, dates are read as text, so you cannot subtract them or extract the month.

In [4]:
print(pd.read_csv(RAW + "patients.csv")["enrolled"].dtype)
patients = pd.read_csv(RAW + "patients.csv", parse_dates=["enrolled"])
print(patients["enrolled"].dtype)
print(patients["enrolled"].min(), "to", patients["enrolled"].max())

str
datetime64[us]
2025-01-02 00:00:00 to 2025-12-31 00:00:00


### Missing value codes

The sample register uses codes like `not done` and blanks. Telling pandas about them while reading
turns them into proper missing values (NaN):

In [5]:
register = pd.read_csv(RAW + "sample_register_raw.csv", na_values=["not done", ""])
print(register["concentration"].isna().sum(), "missing concentrations")
register.head()

44 missing concentrations


,sample_id,patient_id,collected,sample_type,result,concentration,temperature_c,technician
0,S-0001,P0285,14/07/2025,blood,Negative,>100,4.2,BK
1,S-0002,P0225,01/05/2025,urine,Negative,3.2,5.7,CM
2,S-0003,P0067,05/03/2025,urine,Negative,22.0,2.4,BK
3,S-0004,P0218,10/01/2025,BLOOD,NEG,8.0,3.9,BK
4,S-0005,P0195,14/12/2025,BLOOD,NEG,8.0,4.8,BK


The concentration column is still text, because it contains values like `<0.5` and `>100`. That is
handled in notebook 05.

## TSV and other separators

In [6]:
expr = pd.read_csv(RAW + "gene_expression.tsv", sep="\t", index_col="sample")
print(expr.shape)
expr.iloc[:5, :6]

(24, 21)


,condition,GENE01,GENE02,GENE03,GENE04,GENE05
sample,,,,,,
S01,control,8.027,7.850,8.401,7.994,7.130
S02,control,9.317,8.264,7.915,10.016,7.846
S03,control,7.270,9.897,8.571,7.260,6.094
S04,control,8.792,8.054,9.123,7.626,6.209
S05,control,7.067,8.583,8.175,9.287,9.174


Files without a header row need `header=None` and `names=`:

In [7]:
seqs = pd.read_csv(RAW + "sequences.txt", sep="\t", header=None, names=["name", "sequence"])
seqs["length"] = seqs["sequence"].str.len()
seqs.head()

,name,sequence,length
0,seq01,GGTCGTCTGAGTTAAGCAAGAGGACGCCATAGGCTGATCTAGGTGG...,85
1,seq02,GATGGAAGCGAAGCCACCTCGCCGTTCCACCTGATCTCTCCCACTA...,93
2,seq03,AAGATGGGTGGGCGAGATGCGTCTCCTCACAACTTCGTTGGTACTC...,76
3,seq04,TGCTCCAGCGTTTAGACTTAAGGGAAGCTGGGGAGTGGGTTTGTAG...,117
4,seq05,GGCTACGTGCCACACCGGCGTGTAAAAGTATCGCATGGCTCCAGAT...,72


## Excel files

`read_excel` needs the `openpyxl` package for `.xlsx` files. `sheet_name` chooses the sheet; `None`
reads every sheet into a dictionary.

In [8]:
sites = pd.read_excel(RAW + "sites.xlsx", sheet_name="sites")
sites

,site,region,beds,opened
0,Site A,North,120,1998
1,Site B,Central,340,1972
2,Site C,Central,210,2005
3,Site D,South,80,2014


In [9]:
all_sheets = pd.read_excel(RAW + "sites.xlsx", sheet_name=None)
print(list(all_sheets))
all_sheets["staff"].head()

['sites', 'staff']


,site,role,years
0,Site D,Lab,1
1,Site A,Lab,16
2,Site C,Nurse,24
3,Site B,Nurse,18
4,Site D,Doctor,12


## Other formats, briefly

| format | read | write |
|---|---|---|
| JSON | `pd.read_json` | `df.to_json` |
| Parquet (fast, keeps types) | `pd.read_parquet` | `df.to_parquet` |
| SQL databases | `pd.read_sql` | `df.to_sql` |
| Stata, SPSS, SAS | `read_stata`, `read_spss`, `read_sas` | `to_stata` |
| clipboard | `pd.read_clipboard` | `df.to_clipboard` |

## Reading from the web

`read_csv` also accepts a URL. Seaborn ships example datasets this way, which we use in later modules:

```python
pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv")
```

## Exporting

Results and cleaned data go to `datasets/processed`, never back over the raw file.

In [10]:
summary = patients.groupby("site").agg(n=("patient_id", "count"), mean_age=("age", "mean")).round(1)
summary

,n,mean_age
site,,
Site A,111,49.6
Site B,87,56.0
Site C,54,50.8
Site D,48,48.2


In [11]:
from pathlib import Path
Path(PROCESSED).mkdir(exist_ok=True)

summary.to_csv(PROCESSED + "age_by_site.csv")
patients.to_csv(PROCESSED + "patients_copy.csv", index=False)     # index=False: do not write 0, 1, 2 ...
print(pd.read_csv(PROCESSED + "age_by_site.csv"))

     site    n  mean_age
0  Site A  111      49.6
1  Site B   87      56.0
2  Site C   54      50.8
3  Site D   48      48.2


Several tables into one Excel workbook:

In [12]:
with pd.ExcelWriter(PROCESSED + "patients_summary.xlsx") as writer:
    summary.to_excel(writer, sheet_name="by_site")
    patients.groupby("group")["age"].describe().round(1).to_excel(writer, sheet_name="by_group")
print(pd.ExcelFile(PROCESSED + "patients_summary.xlsx").sheet_names)

['by_site', 'by_group']


In [13]:
for f in ["age_by_site.csv", "patients_copy.csv", "patients_summary.xlsx"]:
    Path(PROCESSED + f).unlink()           # tidy up the demo files

## Practise

Exercises for this folder are in `exercises/Pandas_Exercises.ipynb`.

## Further reading

- [pandas user guide](https://pandas.pydata.org/docs/user_guide/index.html)
- [IO tools: reading and writing](https://pandas.pydata.org/docs/user_guide/io.html)
- [read_csv reference](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html)